# UMT-RSIAT on Google Colab
Notebook này chạy code trên SSD tạm của Colab, còn CIFAR-100, checkpoint và log được lưu bền vững trong Google Drive. Trước khi chạy, hãy commit và push phần code UMT-RSIAT lên GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = 'https://github.com/PhThuan-tech/RSIAT.git'
BRANCH = 'main'
PROJECT_DIR = Path('/content/RSIAT')
OUTPUT_ROOT = Path('/content/drive/MyDrive/RSIAT_runs')
PERSISTENT_DATA_DIR = Path('/content/drive/MyDrive/RSIAT_data/datasets')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PERSISTENT_DATA_DIR.mkdir(parents=True, exist_ok=True)

if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

# Nếu runtime hiện tại đã tải dataset vào repo, sao chép nó lên Drive trước.
local_data_dir = PROJECT_DIR / 'data' / 'datasets'
if local_data_dir.is_symlink():
    if local_data_dir.resolve() != PERSISTENT_DATA_DIR.resolve():
        local_data_dir.unlink()
elif local_data_dir.exists():
    if any(local_data_dir.iterdir()):
        print('Copying existing datasets to Google Drive...')
        shutil.copytree(local_data_dir, PERSISTENT_DATA_DIR, dirs_exist_ok=True)
    shutil.rmtree(local_data_dir)

local_data_dir.parent.mkdir(parents=True, exist_ok=True)
if not local_data_dir.exists():
    local_data_dir.symlink_to(PERSISTENT_DATA_DIR, target_is_directory=True)

os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())
print('Persistent datasets:', PERSISTENT_DATA_DIR)
print('Persistent outputs:', OUTPUT_ROOT)
print('Dataset link:', local_data_dir, '->', local_data_dir.resolve())

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# Colab đã cài PyTorch/CUDA. Cài các dependency còn lại của repo.
%pip install -q -r requirements.txt

In [ ]:
# Chuẩn bị CIFAR-100 một lần trong Google Drive.
from torchvision.datasets import CIFAR100

try:
    cifar_train = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=True, download=False)
    cifar_test = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=False, download=False)
    print('CIFAR-100 found in Drive; download skipped.')
except RuntimeError:
    print('CIFAR-100 is incomplete or missing; downloading once to Drive...')
    cifar_train = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=True, download=True)
    cifar_test = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=False, download=True)

print('CIFAR train/test:', len(cifar_train), len(cifar_test))
print('Saved at:', PERSISTENT_DATA_DIR)

# Tạo config Colab: chỉ chạy một seed và lưu output vào Google Drive.
import json
from pathlib import Path

def make_colab_config(source, destination, seed=1993, resume=False):
    config = json.loads(Path(source).read_text())
    config['seed'] = [seed]
    config['resume'] = resume
    config['output_root'] = str(OUTPUT_ROOT)
    Path(destination).write_text(json.dumps(config, indent=2))

make_colab_config('exps/umt_adapter_cifar224_smoke.json', '/content/umt_smoke.json')
make_colab_config('exps/umt_adapter_cifar224.json', '/content/umt_full.json')

# Test nhanh projector và loss, không tải dataset và không train ViT.
!python -m unittest tests.test_research_components

## Smoke test
CIFAR-100 được đọc từ `MyDrive/RSIAT_data/datasets` thông qua liên kết `data/datasets`. Sau lần tải thành công đầu tiên, các runtime Colab sau sẽ dùng lại bản trên Drive. Smoke config chạy 1 epoch mỗi stage để kiểm tra toàn bộ pipeline; đây không phải kết quả dùng để báo cáo.

In [ ]:
!python main.py --config /content/umt_smoke.json

## Full CIFAR-100 experiment
Cấu hình đầy đủ chạy ba class order `[1993, 2024, 3407]`. Nếu muốn chạy từng seed để phù hợp giới hạn phiên Colab, sửa trường `seed` trong JSON thành một phần tử và bật `resume: true` khi tiếp tục đúng run đó.

In [ ]:
# Bỏ dấu # ở dòng dưới sau khi smoke test thành công.
# !python main.py --config /content/umt_full.json

In [ ]:
# Xem các log và checkpoint đã lưu trên Drive.
from pathlib import Path
print('Recent logs:')
for path in sorted((OUTPUT_ROOT / 'logs/umt_adapter').rglob('*.log'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)
print('Recent checkpoints:')
for path in sorted((OUTPUT_ROOT / 'ckpt/umt_weak_moment_topk').rglob('task_*.pkl'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)